# Kiểm tra bộ dữ liệu DiverseVul

Kiểm tra các bản ghi C/C++ ở mức hàm để đánh giá mức phù hợp cho bài toán phân loại CWE. Notebook không tự chọn số CWE phổ biến nhất, chính sách nhãn, cách chia dữ liệu cuối cùng, biểu diễn đồ thị hoặc kiến trúc GNN.

## 1. Thiết lập

Xác định mọi đường dẫn từ thư mục gốc repository và cố định seed dùng để lấy mẫu.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_dataset
from src.data.diversevul_audit import run_audit, read_optional_metadata, metadata_audit, read_label_noise_summary, schema_table
from src.utils.config import load_config

SEED = 105
config = load_config(ROOT / "configs/data/diversevul.yaml")["dataset"]
raw_dir = ROOT / config["local_raw_dir"]
print("Repository:", ROOT)
print("Thư mục dữ liệu thô:", raw_dir)

Repository: D:\PROJECT\software-bug-detection-using-graphs
Thư mục dữ liệu thô: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul


## 2. Nạp dữ liệu

Nạp đầy đủ bản dữ liệu local thực tế. Nếu thiếu dữ liệu, trạng thái bị chặn phải được hiển thị tường minh.

In [2]:
dataset_available = False
blocked_reason = None
try:
    loaded = load_dataset(config["name"], raw_dir)
    raw = loaded.frame
    dataset_available = True
    print("File:", loaded.source_file)
    print("Shape:", raw.shape)
except FileNotFoundError as error:
    blocked_reason = str(error)
    print("BLOCKED:", blocked_reason)

File: D:\PROJECT\software-bug-detection-using-graphs\data\raw\diversevul\diversevul.jsonl
Shape: (330492, 8)


## 3. Cấu trúc dữ liệu

In các cột, kiểu dữ liệu, kích thước, ánh xạ field, dữ liệu thiếu và ví dụ giới hạn từ dữ liệu thật. Không tự tạo field.

In [3]:
if dataset_available:
    result = run_audit(raw, config)
    print(raw.dtypes)
    print("Các field đã ánh xạ:", result.fields)
    display(result.tables["missing_values"])
else:
    print("Chưa quan sát được schema vì không có file dataset nào được load.")

func            str
target        int64
cwe          object
project         str
commit_id       str
hash         object
size          int64
message         str
dtype: object
Các field đã ánh xạ: {'sample_id': None, 'source_code': 'func', 'is_vulnerable': 'target', 'cwe': 'cwe', 'project': 'project', 'commit': 'commit_id', 'provided_hash': 'hash', 'cve': None, 'repository': None}


,field,meaning,dtype,missing_count,missing_percentage,example
0,func,source_code,str,0,0.0,int _gnutls_ciphertext2compressed(gnutls_sessi...
1,target,is_vulnerable,int64,0,0.0,1
2,cwe,cwe,object,0,0.0,[]
3,project,project,str,0,0.0,gnutls
4,commit_id,commit,str,0,0.0,7ad6162573ba79a4392c63b453ad0220ca6c5ace
5,hash,provided_hash,object,0,0.0,73008646937836648589283922871188272089
6,size,field thô chưa được ánh xạ,int64,0,0.0,157
7,message,field thô chưa được ánh xạ,str,0,0.0,added an extra check while checking the padding.
8,KHÔNG TỒN TẠI,sample_id,N/A,330492,100.0,
9,KHÔNG TỒN TẠI,cve,N/A,330492,100.0,


## 4. Xác minh thống kê trong paper

Tính lại sáu metric chính và giữ nguyên mọi chênh lệch so với baseline trong paper.

In [4]:
display(result.tables['paper_comparison']) if dataset_available else print('Chưa tính được.')

,metric,paper,dataset_actual,difference
0,total_samples,349437,330492,-18945
1,vulnerable_samples,18945,18945,0
2,non_vulnerable_samples,330492,311547,-18945
3,unique_projects,797,800,3
4,unique_commits,7514,7653,139
5,unique_cwe,150,150,0


## 5. Dữ liệu bị thiếu

Kiểm tra missingness của từng cột thô, bao gồm các semantic field không tồn tại.

In [5]:
display(result.tables['missing_values']) if dataset_available else print('Chưa tính được.')

,field,meaning,dtype,missing_count,missing_percentage,example
0,func,source_code,str,0,0.0,int _gnutls_ciphertext2compressed(gnutls_sessi...
1,target,is_vulnerable,int64,0,0.0,1
2,cwe,cwe,object,0,0.0,[]
3,project,project,str,0,0.0,gnutls
4,commit_id,commit,str,0,0.0,7ad6162573ba79a4392c63b453ad0220ca6c5ace
5,hash,provided_hash,object,0,0.0,73008646937836648589283922871188272089
6,size,field thô chưa được ánh xạ,int64,0,0.0,157
7,message,field thô chưa được ánh xạ,str,0,0.0,added an extra check while checking the padding.
8,KHÔNG TỒN TẠI,sample_id,N/A,330492,100.0,
9,KHÔNG TỒN TẠI,cve,N/A,330492,100.0,


## 6. Mức bao phủ CWE của dữ liệu có lỗ hổng

Phân tích cho bài toán phân loại CWE chỉ sử dụng các bản ghi có lỗ hổng.

In [6]:
display(result.tables['dataset_summary']) if dataset_available else print('Chưa tính được.')

,metric,value
0,total_samples,330492.000000
1,vulnerable_samples,18945.000000
2,non_vulnerable_samples,311547.000000
3,unique_projects,800.000000
4,unique_commits,7653.000000
5,unique_cwe,150.000000
6,vulnerable_with_cwe,16109.000000
7,vulnerable_without_cwe,2836.000000
8,vulnerable_cwe_coverage_percentage,85.030351


## 7. Phân bố CWE

Tính mức bao phủ theo hàm, dự án, commit và dữ liệu có lỗ hổng đã gắn nhãn cho từng CWE. Bản ghi có nhiều CWE đóng góp vào từng CWE được gắn.

In [7]:
display(result.tables['cwe_distribution'].head(20)) if dataset_available else print('Chưa tính được.')

,cwe,sample_count,percentage_of_labeled_vulnerable,project_count,commit_count
0,CWE-787,2896,17.977528,277,1081
1,CWE-125,1869,11.602210,176,819
2,CWE-119,1633,10.137190,192,720
3,CWE-20,1315,8.163139,148,499
4,CWE-703,1228,7.623068,152,529
5,CWE-416,1005,6.238749,79,392
6,CWE-476,975,6.052517,125,503
7,CWE-190,783,4.860637,96,293
8,CWE-200,747,4.637159,102,369
9,CWE-399,509,3.159724,59,240


## 8. Phân tích ngưỡng

Đánh giá các ngưỡng 20/50/100/200 mẫu trên mỗi CWE mà không tự chọn ngưỡng.

In [8]:
display(result.tables['threshold_analysis']) if dataset_available else print('Chưa tính được.')

,threshold,number_of_cwe,number_of_samples,percentage_samples_retained,minimum_class_size,maximum_class_size
0,20,76,15814,98.168726,20,2896
1,50,42,14985,93.022534,51,2896
2,100,31,14581,90.514619,108,2896
3,200,21,13555,84.145509,201,2896


## 9. Phân tích mẫu có nhiều CWE

Đo số mẫu có 0, 1 hoặc từ 2 CWE trở lên mà không tự chuyển bài toán thành phân loại một nhãn.

In [9]:
if dataset_available:
    display(result.tables["multi_cwe_summary"])
    display(result.tables["cwe_count_distribution"])
    display(result.tables["multi_cwe_examples"])
else:
    print("Chưa tính được.")

,category,sample_count,percentage_of_vulnerable
0,zero_cwe,2836,14.969649
1,exactly_one_cwe,11894,62.781737
2,two_or_more_cwe,4215,22.248614


,number_of_cwe_per_sample,sample_count
0,0,2836
1,1,11894
2,2,3763
3,3,422
4,4,30


,sample_id,project,commit,cwe,code_preview
553,row-553,nbd,3ef52043861ab16352d49af89e048ba6339d6df8,CWE-119|CWE-787,int mainloop(CLIENT *client) { \tstruct nbd_re...
613,row-613,rsyslog,1ef709cc97d54f74d3fdeb83788cc4b01f4c6a2a,CWE-772|CWE-401,processBatchMultiRuleset(batch_t *pBatch) { \t...
614,row-614,rsyslog,1ef709cc97d54f74d3fdeb83788cc4b01f4c6a2a,CWE-772|CWE-401,"batchInit(batch_t *pBatch, int maxElem) { \tDE..."
615,row-615,rsyslog,1ef709cc97d54f74d3fdeb83788cc4b01f4c6a2a,CWE-772|CWE-401,"batchCopyElem(batch_obj_t *pDest, batch_obj_t ..."
618,row-618,mono,722f9890f09aadfc37ae479e7d946d5fc5ef7b91,CWE-399|CWE-264,void ves_icall_System_Threading_InternalThread...
620,row-620,mono,2f00e4bbb2137130845afb1b2a1e678552fc8e5c,CWE-362|CWE-264,ves_icall_System_Array_FastCopy (MonoArray *so...
621,row-621,mono,3f8ee42b8c867d9a4c18c22657840d072cca5c3a,CWE-399|CWE-264,void mono_reflection_destroy_dynamic_method (M...
622,row-622,mono,3f8ee42b8c867d9a4c18c22657840d072cca5c3a,CWE-399|CWE-264,void mono_reflection_create_dynamic_method (M...
623,row-623,mono,3f8ee42b8c867d9a4c18c22657840d072cca5c3a,CWE-399|CWE-264,void mono_reflection_create_dynamic_method (M...
624,row-624,mono,89d1455a80ef13cddee5d79ec00c06055da3085c,CWE-399|CWE-264,mono_runtime_shutdown (void) { \tmono_domain_f...


## 10. Phân bố dự án

So sánh tổng số hàm, số hàm có lỗ hổng và độ rộng CWE trên từng dự án.

In [10]:
display(result.tables['project_distribution'].head(20)) if dataset_available else print('Chưa tính được.')

,project,total_samples,vulnerable_samples,number_of_cwe
0,linux,72270,3366,75
1,linux-2.6,10665,449,20
2,php-src,9041,969,38
3,server,8101,102,13
4,qemu,7493,470,29
5,gpac,6797,150,14
6,openssl,6239,664,29
7,vim,6126,223,22
8,envoy,5849,257,12
9,tensorflow,5056,629,28


## 11. Phân tích CWE theo dự án

Đo số dự án hỗ trợ và mức tập trung vào dự án lớn nhất cho từng CWE.

In [11]:
if dataset_available:
    display(result.tables["cwe_project_distribution"].head(30))
    display(result.tables["project_split_feasibility"])
else:
    print("Chưa tính được.")

,cwe,sample_count,project_count,largest_project_sample_count,largest_project_share
0,CWE-787,2896,277,401,13.846685
1,CWE-125,1869,176,310,16.586410
2,CWE-119,1633,192,279,17.085119
3,CWE-20,1315,148,249,18.935361
4,CWE-703,1228,152,408,33.224756
5,CWE-416,1005,79,434,43.184080
6,CWE-476,975,125,229,23.487179
7,CWE-190,783,96,127,16.219668
8,CWE-200,747,102,214,28.647925
9,CWE-399,509,59,147,28.880157


,minimum_projects_per_cwe,number_of_cwe,number_of_cwe_with_at_least_20_samples,number_of_cwe_with_at_least_50_samples,number_of_cwe_with_at_least_100_samples,number_of_cwe_with_at_least_200_samples
0,2,114,74,42,31,21
1,3,97,72,42,31,21
2,5,73,66,42,31,21
3,10,51,49,38,30,21


## 12. Phân tích dữ liệu trùng lặp

So sánh mã băm của mã nguồn nguyên bản với mã băm sau khi chuẩn hóa bảo thủ ký tự xuống dòng và khoảng trắng cuối dòng, đồng thời đếm xung đột nhãn.

In [12]:
display(result.tables['duplicate_summary']) if dataset_available else print('Chưa tính được.')

,level,records_with_code,unique_code,duplicate_groups,duplicate_affected_records,duplicate_excess_records,duplicate_affected_percentage,vulnerable_label_conflict_groups,cwe_conflict_groups
0,raw_source,330491,330491,0,0,0,0.000000,0,0
1,normalized_source,330491,329628,863,1726,863,0.522253,459,299


## 13. Chất lượng mã nguồn

Báo cáo mã nguồn bị thiếu/rỗng và các phân vị độ dài theo ký tự/dòng. Chủ động không tính token khi chưa chọn bộ tách token.

In [13]:
if dataset_available:
    display(result.tables["source_quality_summary"])
    display(result.tables["function_length_summary"])
    display(result.tables["source_syntax_profile"])
    display(result.tables["source_inspection_sample"])
else:
    print("Chưa tính được.")

,check,sample_count,definition
0,missing_source,0,giá trị source thô là null
1,empty_source,1,"source là null, rỗng hoặc chỉ có whitespace"
2,very_short_source,5608,ít hơn 20 ký tự hoặc 3 dòng (chỉ là cờ sàng lọc)
3,very_long_source,4876,nhiều hơn 10.000 ký tự hoặc 500 dòng (chỉ là c...


,metric,min,median,mean,p90,p95,p99,max
0,characters_per_function,0,516.0,1313.859761,2743.9,4509.45,12890.09,484356
1,lines_per_function,0,19.0,43.582610,91.0,146.00,392.00,24047


,check,sample_count,percentage_of_all_samples,note
0,function_like_delimiters,326289,98.728260,"heuristic lexical, không phải kết quả parse"
1,preprocessor_directive,29947,9.061339,"heuristic lexical, không phải kết quả parse"
2,cpp_specific_marker,44420,13.440567,"heuristic lexical, không phải kết quả parse"


,sample_id,project,commit,cwe,code_preview
16980,row-16980,tcpdump,7335163a6ef82d46ff18f3e6099a157747241629,CWE-125|CWE-787,"handle_mlppp(netdissect_options *ndo, ..."
16860,row-16860,git,684dd4c2b414bcf648505e74498a608f28de4592,CWE-59|CWE-61,int mingw_rmdir(const char *pathname) { \tint ...
16038,row-16038,deepin-clone,e079f3e2712b4f8c28e3e63e71ba1a1f90fce1ab,CWE-59|CWE-61,bool Helper::getPartitionSizeInfo(const QStrin...
16335,row-16335,envoy,ce0ae309057a216aba031aff81c445c90c6ef145,CWE-416,void Filter::onDownstreamEvent(Network::Connec...
14044,row-14044,tensorflow,ca8c013b5e97b1373b3bb1c97ea655e69f31a575,CWE-703|CWE-197,"static void Launch(OpKernelContext* context,..."
17827,row-17827,openssl,ee2ffc279417f15fef3b1073c7dc81a908991516,,"int MAIN(int argc, char **argv) \t{ \tunsigned..."
5435,row-5435,yara,3119b232c9c453c98d8fa8b6ae4e37ba18117cd4,CWE-476|CWE-703|CWE-125,\twhile ( /*CONSTCOND*/1 )\t\t/* loops until ...
995,row-995,tor,62637fa22405278758febb1743da9af562524d4c,CWE-399,"dns_resolve_impl(edge_connection_t *exitconn, ..."
2759,row-2759,qemu,3251bdcf1c67427d964517053c3d185b46e618e8,CWE-399,"void ide_dma_cb(void *opaque, int ret) { I..."
14173,row-14173,libtpms,5cc98a62dc6f204dcf5b87c2ee83ac742a6a319b,CWE-787,"TPMI_ALG_SYM_Unmarshal(TPMI_ALG_SYM *target, B..."


## 14. Mẫu kiểm tra mức sẵn sàng để tạo đồ thị

Chỉ tạo danh mục nhỏ với seed cố định để thử Joern ở giai đoạn sau; không tạo CPG trong notebook này.

In [14]:
if dataset_available:
    display(result.tables["graph_sample_manifest"])
else:
    print("Không thể tạo manifest khi chưa có dataset thô.")

,sample_id,project,commit,cwe,label,source_reference
17543,row-17543,acrn-hypervisor,2b3dedfb9ba13f15887f22b935d373f36c9a59fa,CWE-617|CWE-703,vulnerable,dataset row sample_id=row-17543
13809,row-13809,linux,05ab8f2647e4221cbdb3856dd7d32bd5407316b3,CWE-416|CWE-125|CWE-189,vulnerable,dataset row sample_id=row-13809
12897,row-12897,rizin,05bbd147caccc60162d6fba9baaaf24befa281cd,CWE-200|CWE-787,vulnerable,dataset row sample_id=row-12897
4824,row-4824,jasper,d42b2388f7f8e0332c846675133acea151fc557a,CWE-20|CWE-190,vulnerable,dataset row sample_id=row-4824
8688,row-8688,envoy,afc39bea36fd436e54262f150c009e8d72db5014,CWE-400|CWE-703,vulnerable,dataset row sample_id=row-8688
12023,row-12023,linux,3e3b5dfcd16a3e254aab61bd1e8c417dd4503102,,vulnerable,dataset row sample_id=row-12023
8932,row-8932,libnbd,2c1987fc23d6d0f537edc6d4701e95a2387f7917,,vulnerable,dataset row sample_id=row-8932
16198,row-16198,re2c,039c18949190c5de5397eba504d2c75dad2ea9ca,CWE-787,vulnerable,dataset row sample_id=row-16198
567,row-567,pango,336bb3201096bdd0494d29926dd44e8cca8bed26,,vulnerable,dataset row sample_id=row-567
13727,row-13727,folly,c321eb588909646c15aefde035fd3133ba32cdee,CWE-125,vulnerable,dataset row sample_id=row-13727


## 15. Kết quả

Sinh tất cả bảng, biểu đồ, danh mục mẫu, kết quả kiểm tra metadata, bảng label-noise và báo cáo Markdown từ cùng một kết quả đã tính.

In [15]:
import subprocess
completed = subprocess.run(
    [sys.executable, str(ROOT / "scripts/run_diversevul_eda.py")],
    cwd=ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout.strip())
if dataset_available:
    metadata, _ = read_optional_metadata(raw_dir)
    result.tables["metadata_audit"] = metadata_audit(result.records, metadata)
    label_noise, _ = read_label_noise_summary(raw_dir)
    result.tables["label_noise_summary"] = label_noise
    display(result.tables["metadata_audit"])
    display(result.tables["label_noise_summary"])
    print(result.findings)
else:
    print("Không có kết quả EDA nào được tuyên bố; các artifact được sinh đều ghi rõ trạng thái BLOCKED.")

EDA hoÃ n táº¥t: D:\PROJECT\software-bug-detection-using-graphs\reports\dataset\dataset_eda.md


,metric,value,note
0,metadata_rows,7511,
1,dataset_unique_commits,7653,
2,metadata_unique_commits,7511,
3,joined_unique_commits,7169,Phần giao của commit ID
4,unjoined_dataset_commits,484,
5,unjoined_metadata_commits,342,
6,joined_dataset_records,308696,Record có commit ID xuất hiện trong metadata
7,unjoined_dataset_records,21796,
8,missing_commit_url_rows,0,
9,missing_repo_url_rows,1,


,dataset,sample_size,correct_label,vulnerability_spread_multiple_functions,relevant_but_not_vulnerable,irrelevant,correct_percentage
0,DiverseVul,50,30,5,6,9,60.000000
1,Union of Three,50,18,6,6,20,36.000000
2,CVEFixes,29,15,3,5,6,51.724138
3,BigVul,32,8,5,3,16,25.000000
4,CrossVul,23,11,3,5,4,47.826087


{'total_samples': 330492, 'vulnerable_samples': 18945, 'non_vulnerable_samples': 311547, 'unique_projects': 800, 'unique_commits': 7653, 'unique_cwe': 150, 'raw_column_count': 8, 'vulnerable_with_cwe': 16109, 'vulnerable_without_cwe': 2836, 'multi_cwe_samples': 4215, 'unique_cwe_vulnerable': 150}


## 16. Các quyết định nghiên cứu còn mở

Bằng chứng hỗ trợ nhưng không tự chốt các quyết định dưới đây.

In [16]:
decisions = [
    "Top-K hoặc số sample tối thiểu trên mỗi CWE",
    "cách xử lý single-label, multi-label, hierarchical hoặc ambiguous sample",
    "project-aware split cuối cùng",
    "chính sách xử lý duplicate và label conflict",
    "graph representation và cấu hình Joern",
    "kiến trúc GNN và evaluation protocol",
]
for decision in decisions:
    print("-", decision)

- Top-K hoặc số sample tối thiểu trên mỗi CWE
- cách xử lý single-label, multi-label, hierarchical hoặc ambiguous sample
- project-aware split cuối cùng
- chính sách xử lý duplicate và label conflict
- graph representation và cấu hình Joern
- kiến trúc GNN và evaluation protocol
